# 21 - Random Forest with three seeds

How much do the Random Forest results depend on the random seed? Same protocol as notebook 15, run with three seeds on HDFS and BGL.

**Seeds: 42, 7, 123**, the same as the Isolation Forest study (notebook 20). 42 is the seed the code uses now (`SEED` in `src/models/supervised.py`). The seed goes to both the Random Forest and SMOTE.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.models.supervised import METHODS, evaluate_rf, train_rf
from src.models.unsupervised import time_split_val

DOCS, PROCESSED = ROOT / "docs", ROOT / "data" / "processed"
SEEDS = [42, 7, 123]
LABELLED = {"HDFS": ("hdfs", "order"), "BGL": ("bgl", "window")}
METRICS = ["precision", "recall", "f1", "false_alarm_rate"]

## Runs

Random Forest as now: 100 trees, unlimited depth. For each seed, the three imbalance settings (none, balanced class weights, SMOTE on the training part only) are trained on the first 60%, the best by F1 on the next 10% is chosen (the first one on ties), and only that one is tested on the last 30%. This is `select_rf_method` from notebook 15 with the seed passed to `train_rf`.

In [2]:
rows, val_rows = [], []
for name, (prefix, order_col) in LABELLED.items():
    counts = pd.read_parquet(PROCESSED / f"{prefix}_counts.parquet")
    event_cols = list(counts.filter(regex=r"^E\d+$").columns)
    fit, val, test = time_split_val(counts, order_col=order_col)
    for seed in SEEDS:
        models, val_f1 = {}, {}
        for method in METHODS:
            models[method], _ = train_rf(fit[event_cols], fit["Label"], method=method, random_state=seed)
            val_f1[method] = evaluate_rf(models[method], val[event_cols], val["Label"])["f1"]
            val_rows.append({"dataset": name, "seed": seed, "setting": method, "val_f1": val_f1[method]})
        best = max(METHODS, key=lambda m: val_f1[m])  # first method on ties, as in select_rf_method
        m = evaluate_rf(models[best], test[event_cols], test["Label"])
        rows.append({"dataset": name, "seed": seed, "chosen_setting": best, **{k: m[k] for k in METRICS}})

results = pd.DataFrame(rows)
results.round(4).to_csv(DOCS / "seed_results_rf.csv", index=False)
results.style.format({m: "{:.4f}" for m in METRICS}).hide(axis="index")

dataset,seed,chosen_setting,precision,recall,f1,false_alarm_rate
HDFS,42,balanced,1.0000,0.9970,0.9985,0.0000
HDFS,7,none,1.0000,0.9979,0.9989,0.0000
HDFS,123,none,1.0000,0.9979,0.9989,0.0000
BGL,42,balanced,0.9551,0.0721,0.1341,0.0003
BGL,7,balanced,0.9585,0.0740,0.1374,0.0003
BGL,123,balanced,0.9552,0.0723,0.1344,0.0003


Validation F1 of each setting, per seed:

In [3]:
pd.DataFrame(val_rows).pivot_table(index=["dataset", "seed"], columns="setting", values="val_f1", sort=False)[METHODS].round(4)

setting         none  balanced   smote
dataset seed                          
HDFS    42    0.9985    0.9987  0.9934
        7     0.9987    0.9987  0.9934
        123   0.9987    0.9987  0.9936
BGL     42    0.1719    0.2043  0.1999
        7     0.1727    0.2058  0.1982
        123   0.1740    0.2035  0.2015

## Mean and standard deviation over the seeds

Sample standard deviation (n - 1). The reported values are the seed-42 results from notebook 15.

In [4]:
summary = results.groupby("dataset", sort=False)[METRICS].agg(["mean", "std"])
summary.columns = [f"{m}_{s}" for m, s in summary.columns]
summary = summary.reset_index()
reported = pd.read_csv(DOCS / "cross_system_results.csv").query("model == 'Random Forest' and dataset == `chosen on`")
summary["reported_f1"] = summary["dataset"].map(reported.set_index("dataset")["f1"])
summary.style.format({c: "{:.4f}" for c in summary.columns if c != "dataset"}).hide(axis="index")

dataset,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,false_alarm_rate_mean,false_alarm_rate_std,reported_f1
HDFS,1.0000,0.0000,0.9976,0.0005,0.9988,0.0002,0.0000,0.0000,0.9985
BGL,0.9563,0.0019,0.0728,0.0010,0.1353,0.0018,0.0003,0.0000,0.1341


In [5]:
pd.DataFrame({m: summary.set_index("dataset").apply(lambda r: f"{r[m + '_mean']:.3f} ± {r[m + '_std']:.3f}", axis=1)
              for m in METRICS})

,precision,recall,f1,false_alarm_rate
dataset,,,,
HDFS,1.000 ± 0.000,0.998 ± 0.000,0.999 ± 0.000,0.000 ± 0.000
BGL,0.956 ± 0.002,0.073 ± 0.001,0.135 ± 0.002,0.000 ± 0.000


## Findings

- **Seed 42 gives exactly the reported results** (HDFS F1 0.9985, BGL F1 0.134).
- **The results are stable.** HDFS F1 0.9988 ± 0.0002, BGL F1 0.135 ± 0.002; precision, recall and false alarm rate move by less than 0.002.
- **Chosen setting:** BGL picks balanced class weights with every seed. On HDFS, seed 42 picks balanced and seeds 7 and 123 pick none, but there "none" and "balanced" have the same validation F1 (0.9987), so the choice makes no practical difference (test F1 0.9985 vs 0.9989).
- SMOTE is never chosen: on HDFS it is a little worse on validation, on BGL a little below balanced class weights.